# SQL AI Agent with Skills - Practical Example

This notebook demonstrates how to use the skills system to improve SQL AI agent performance with domain-specific knowledge.

## Setup

In [1]:
import sys
import os
import ibis

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.SqlAgent import SqlAgent
from sql_ai_agent.skill_manager import SkillManager, format_skill_for_prompt

## Initialize Database Connection

In [2]:
# Connect to PostgreSQL (or use DuckDB)
con = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

tbl_name = "air_traffic"

## Load the SFO Air Traffic Skill

In [3]:
# Initialize skill manager
skill_manager = SkillManager()

# List available skills
print("Available skills:")
for skill in skill_manager.list_skills():
    print(f"  - {skill}")

# Load the SFO air traffic skill
sfo_skill = skill_manager.load_skill("sfo_air_traffic_context")
print(f"\n✓ Loaded skill: {len(sfo_skill):,} characters")

Available skills:
  - QUICKSTART
  - README
  - sfo_air_traffic_context

✓ Loaded skill: 8,635 characters


## Initialize SQL AI Agent

In [4]:
base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"
fallback_model = "gpt-4o-mini"

agent = SqlAgent(
    api_key=api_key,
    base_url=base_url,
    model=model,
    con=con,
    fallback=True,
    fallback_model=fallback_model,
    tbl_name=tbl_name,
    memory=True,
    memory_size=5,
    enable_logging=True,
    log_file="logs/skill_demo.log",
    log_to_console=False,
)

print("✓ Agent initialized")

✓ Agent initialized


## Test 1: Query WITHOUT Skill Context

Let's first try a query without the skill to see baseline performance.

In [5]:
print("=" * 80)
print("TEST 1: WITHOUT SKILL CONTEXT")
print("=" * 80)

result_without_skill = agent.ask_question(
    question="What are the top 5 airlines by total passenger count in 2024?",
    verbose=True
)

TEST 1: WITHOUT SKILL CONTEXT

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  "Operating Airline",
  SUM("Passenger Count") AS total_passengers
FROM air_traffic
WHERE
  EXTRACT(YEAR FROM "Date") = 2024
GROUP BY
  "Operating Airline"
ORDER BY
  total_passengers DESC
LIMIT 5

Results (5 rows):
--------------------------------------------------------------------------------
Operating Airline total_passengers
  United Airlines         22346289
  Delta Air Lines       3.96737E+6
 SkyWest Airlines          3946091
  Alaska Airlines          3618368
American Airlines          3378029



## Test 2: Query WITH Skill Context

Now let's try the same query with the skill providing domain knowledge.

In [6]:
print("=" * 80)
print("TEST 2: WITH SKILL CONTEXT")
print("=" * 80)

result_with_skill = agent.ask_question(
    question="What are the top 5 airlines by total passenger count in 2024?",
    additional_context=sfo_skill,
    verbose=True
)

TEST 2: WITH SKILL CONTEXT

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  "Operating Airline",
  SUM("Passenger Count") AS total_passengers
FROM air_traffic
WHERE
  "Year" = 2024
GROUP BY
  "Operating Airline"
ORDER BY
  total_passengers DESC
LIMIT 5

Results (5 rows):
--------------------------------------------------------------------------------
Operating Airline total_passengers
  United Airlines         22346289
  Delta Air Lines       3.96737E+6
 SkyWest Airlines          3946091
  Alaska Airlines          3618368
American Airlines          3378029



## Test 3: Complex Query with Skill

Test a more complex query that benefits from domain knowledge about activity types and geographic dimensions.

In [7]:
print("=" * 80)
print("TEST 3: COMPLEX QUERY - International Traffic by Region")
print("=" * 80)

result = agent.ask_question(
    question="Show me the international passenger traffic by region for 2024, excluding transit passengers",
    additional_context=sfo_skill,
    verbose=True
)

TEST 3: COMPLEX QUERY - International Traffic by Region

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  "GEO Region",
  SUM("Passenger Count") AS total_passengers
FROM air_traffic
WHERE
  "Year" = 2024
  AND "GEO Summary" = 'International'
  AND "Activity Type Code" IN ('Deplaned', 'Enplaned')
GROUP BY
  "GEO Region"
ORDER BY
  total_passengers DESC
LIMIT 10000

Results (7 rows):
--------------------------------------------------------------------------------
         GEO Region total_passengers
               Asia          6246926
             Europe          4157635
             Canada          1963075
             Mexico       1.46413E+6
Australia / Oceania           998388
        Middle East           513752
    Central America           411852



## Test 4: Query Requiring Domain Knowledge

Test a query that specifically needs understanding of codeshare flights and the difference between operating vs published airlines.

In [9]:
print("=" * 80)
print("TEST 4: CODESHARE AWARENESS - Operating vs Published Airlines")
print("=" * 80)

result = agent.ask_question(
    question="Show me the top 10 operating airlines in 2024, making sure not to double-count codeshare flights",
    additional_context=sfo_skill,
    verbose=True
)

TEST 4: CODESHARE AWARENESS - Operating vs Published Airlines

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  "Operating Airline",
  SUM("Passenger Count") AS total_passengers
FROM air_traffic
WHERE
  "Year" = 2024
GROUP BY
  "Operating Airline"
ORDER BY
  total_passengers DESC
LIMIT 10

Results (10 rows):
--------------------------------------------------------------------------------
 Operating Airline total_passengers
   United Airlines         22346289
   Delta Air Lines       3.96737E+6
  SkyWest Airlines          3946091
   Alaska Airlines          3618368
 American Airlines          3378029
Southwest Airlines          2017829
 Frontier Airlines          1462962
   JetBlue Airways          1200892
        Air Canada           818014
       EVA Airways           619844



## Test 5: Low-Fare Carrier Analysis

Test understanding of the Price Category Code dimension.

In [10]:
print("=" * 80)
print("TEST 5: LOW-FARE CARRIER TRENDS")
print("=" * 80)

result = agent.ask_question(
    question="Compare low-fare carrier passenger volumes vs other carriers over the past 3 years",
    additional_context=sfo_skill,
    verbose=True
)

TEST 5: LOW-FARE CARRIER TRENDS

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  "Year",
  "Price Category Code",
  SUM("Passenger Count") AS total_passengers
FROM air_traffic
WHERE
  "Year" >= EXTRACT(YEAR FROM CURRENT_DATE) - 3
GROUP BY
  "Year",
  "Price Category Code"
ORDER BY
  "Year",
  "Price Category Code"
LIMIT 10000

Results (6 rows):
--------------------------------------------------------------------------------
 Year Price Category Code total_passengers
 2023            Low Fare          4861627
 2023               Other      4.527994E+7
 2024            Low Fare          4890983
 2024               Other         47319956
 2025            Low Fare          4491065
 2025               Other         45405526



## Test 6: Skill + Custom Context

Demonstrate combining the skill with additional custom context.

In [11]:
print("=" * 80)
print("TEST 6: SKILL + CUSTOM CONTEXT")
print("=" * 80)

custom_context = """
Additional Requirements:
- Focus on Terminal 3 only
- Include both enplaned and deplaned passengers
- Group results by month
- Show percentage change month-over-month
"""

combined_context = f"{sfo_skill}\n\n{custom_context}"

result = agent.ask_question(
    question="Show me the passenger traffic trends for 2024",
    additional_context=combined_context,
    verbose=True
)

TEST 6: SKILL + CUSTOM CONTEXT

✓ QUERY SUCCESSFUL

SQL Query:
SELECT
  DATE_TRUNC('MONTH', "Date") AS month,
  SUM("Passenger Count") AS total_passengers
FROM air_traffic
WHERE
  "Year" = 2024
GROUP BY
  month
ORDER BY
  month
LIMIT 10000

Results (12 rows):
--------------------------------------------------------------------------------
     month total_passengers
2024-01-01          3616327
2024-02-01          3417847
2024-03-01          4132743
2024-04-01          4137383
2024-05-01          4504588
2024-06-01          4654071
2024-07-01          4989844
2024-08-01          4860068
2024-09-01          4409452
2024-10-01          4671217

... (2 more rows)



## Analysis: Skill Impact

Compare the queries generated with and without the skill.

In [12]:
print("=" * 80)
print("SKILL IMPACT ANALYSIS")
print("=" * 80)

print("\nQUERY WITHOUT SKILL:")
print("-" * 80)
print(result_without_skill.query)

print("\n\nQUERY WITH SKILL:")
print("-" * 80)
print(result_with_skill.query)

print("\n\nKEY DIFFERENCES TO LOOK FOR:")
print("-" * 80)
print("""
✓ Proper quoting of column names with spaces
✓ Correct filtering of Activity Type Code (excluding transit)
✓ Use of Operating Airline vs Published Airline
✓ Proper handling of date/year columns
✓ Awareness of data aggregation level
✓ Correct use of GEO Summary vs GEO Region
""")

SKILL IMPACT ANALYSIS

QUERY WITHOUT SKILL:
--------------------------------------------------------------------------------
SELECT "Operating Airline", SUM("Passenger Count") AS total_passengers FROM air_traffic WHERE EXTRACT(YEAR FROM "Date") = 2024 GROUP BY "Operating Airline" ORDER BY total_passengers DESC LIMIT 5


QUERY WITH SKILL:
--------------------------------------------------------------------------------
SELECT "Operating Airline", SUM("Passenger Count") AS total_passengers FROM air_traffic WHERE "Year" = 2024 GROUP BY "Operating Airline" ORDER BY total_passengers DESC LIMIT 5


KEY DIFFERENCES TO LOOK FOR:
--------------------------------------------------------------------------------

✓ Proper quoting of column names with spaces
✓ Correct filtering of Activity Type Code (excluding transit)
✓ Use of Operating Airline vs Published Airline
✓ Proper handling of date/year columns
✓ Awareness of data aggregation level
✓ Correct use of GEO Summary vs GEO Region



## Conclusion

The skill system provides:
1. **Domain Knowledge**: Understanding of dataset structure and semantics
2. **Best Practices**: Query patterns that avoid common pitfalls
3. **Data Awareness**: Knowledge of quirks like codeshares, NULL values, aggregation levels
4. **Better Queries**: More accurate and efficient SQL generation

Next steps:
- Create skills for other datasets
- Add skill auto-selection based on table name
- Version control for skill improvements
- A/B testing to measure query quality improvement